# ITASORL on Google Colab — full end-to-end + recorded results

Runs **pytest + all experiments**, records everything under `results/runs/<timestamp>/`,
and downloads **`bundle.zip`** so you can examine outcomes offline.

**Before you start:** Runtime → Change runtime type → **GPU**.

**After the run, read:** `SUMMARY.md` inside the results folder (plain-English verdict).

In [ ]:
# --- Config ---
REPO_URL = "https://github.com/iLevyTate/ITASORL.git"
BRANCH = "main"
REPO_DIR = "/content/ITASORL"

RUN_MODE = "quick"          # "quick" (~30-60 min) | "full" (hours)
SKIP_STEPS = []             # e.g. ["expB2"] to skip longest step

MOUNT_DRIVE = True          # save bundle.zip to Drive (recommended)
DRIVE_RESULTS = "/content/drive/MyDrive/ITASORL_results"
DOWNLOAD_BUNDLE = True      # also trigger browser download of bundle.zip

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=check)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)

In [ ]:
if os.path.isdir(REPO_DIR):
    sh(f"cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
else:
    sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
sh("git log -1 --oneline")

In [ ]:
sh(f"{sys.executable} -m pip install -q --upgrade pip")
dev = os.path.join(REPO_DIR, "requirements-dev.txt")
if os.path.isfile(dev):
    sh(f"{sys.executable} -m pip install -q -r requirements-dev.txt")
else:
    sh(f"{sys.executable} -m pip install -q -r requirements.txt pytest")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
cmd = [sys.executable, "run_e2e.py"]
if RUN_MODE == "quick":
    cmd.append("--quick")
elif RUN_MODE != "full":
    raise ValueError("RUN_MODE must be 'quick' or 'full'")
for step in SKIP_STEPS:
    cmd.extend(["--skip", step])

t0 = time.perf_counter()
subprocess.run(cmd, cwd=REPO_DIR, check=True)
print(f"Wall time: {(time.perf_counter() - t0) / 60:.1f} min")

In [ ]:
latest_ptr = Path(REPO_DIR) / "results" / "LATEST_RUN.txt"
RUN_DIR = Path(latest_ptr.read_text().strip())
BUNDLE = RUN_DIR / "bundle.zip"
SUMMARY = RUN_DIR / "SUMMARY.md"

print("Run directory:", RUN_DIR)
print("Bundle:", BUNDLE, "exists:", BUNDLE.is_file())
print("\n" + "=" * 72)
print(SUMMARY.read_text())
print("=" * 72)

In [ ]:
if MOUNT_DRIVE and BUNDLE.is_file():
    dest = Path(DRIVE_RESULTS) / f"{RUN_DIR.name}_bundle.zip"
    shutil.copy2(BUNDLE, dest)
    print("Saved to Drive:", dest)
    # also copy SUMMARY for quick viewing
    shutil.copy2(SUMMARY, Path(DRIVE_RESULTS) / f"{RUN_DIR.name}_SUMMARY.md")

if DOWNLOAD_BUNDLE and BUNDLE.is_file():
    from google.colab import files
    files.download(str(BUNDLE))
    print("Browser download started for bundle.zip")

## What's in the bundle

| File | Purpose |
|------|---------|
| `SUMMARY.md` | Plain-English outcome — **read this first** |
| `manifest.json` | Step timings, status, artifact index |
| `combined.log` | Full stdout from every step |
| `steps/*.json` | Parsed metrics (AUROC per drift, etc.) |
| `artifacts/` | Figures + `expB2_results.json` |

Unzip locally and open `SUMMARY.md` to decide whether the organism encoded world identity.